# Part 1 — Core Transformer Architecture: A Walkthrough

**Goal of this notebook:** Build every piece of a Transformer block from scratch, so you can read the `.py` files in `part_1/` and understand *why* each line exists.

**What you should know going in:**
- Basic neural-network math (matrix multiplies, gradients, loss)
- A bit about Transformers (heard the names "attention", "GPT", "BERT")
- Comfort with Python and NumPy

**What you'll have at the end:**
- A clear mental map of the Transformer block
- Intuition for *every* tensor shape
- Running code demos that produce real outputs and attention heatmaps

## The Map

Every Transformer block has the same six ingredients stacked in the same order. Below is the "master diagram" — you'll see it many times in this notebook. Each section highlights the ingredient it's about in color, while the rest fade to grey. That way you always know *where you are* in the block.

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]

    pe["1.1 Positional Encoding<br>add position info"]
    ln1["1.5 LayerNorm 1"]
    mha["1.3/1.4 Multi-Head Attention<br>(Q, K, V + causal mask)"]
    add1["+ residual"]
    ln2["1.5 LayerNorm 2"]
    ffn["1.5 Feed-Forward<br>Linear -> GELU -> Linear"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> pe --> ln1 --> mha --> add1
    pe --> add1
    add1 --> ln2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8f4fd,stroke:#2874a6,stroke-width:3px,color:#000
    style pe fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ln1 fill:#e8daef,stroke:#6c3483,stroke-width:3px,color:#000
    style mha fill:#fdebd0,stroke:#b35900,stroke-width:3px,color:#000
    style add1 fill:#fdd,stroke:#c33,stroke-width:3px,color:#000
    style ln2 fill:#e8daef,stroke:#6c3483,stroke-width:3px,color:#000
    style ffn fill:#fff3c4,stroke:#b38600,stroke-width:3px,color:#000
    style add2 fill:#fdd,stroke:#c33,stroke-width:3px,color:#000
    style output fill:#e8f4fd,stroke:#2874a6,stroke-width:3px,color:#000
```

**What to notice in the diagram:**
- **Straight lines** = data flow (left-to-right through the block)
- **Curved lines** = residual connections (`pe -> add1`, `add1 -> add2`) — these are the "gradient highways" that make deep networks trainable
- **Two LayerNorms** = normalize before each sublayer (Pre-Norm design, used by GPT-2/3/4)
- **`+` nodes** = element-wise addition of the residual and the sublayer output

### Color legend (used throughout):

| Color | Role |
|-------|------|
| 🔵 light blue | Input/output tensors |
| 🟢 light green | Position encoding |
| 🟣 light purple | LayerNorm |
| 🟠 light orange | Multi-head attention |
| 🟡 light yellow | Feed-forward |
| 🔴 light pink | Residual `+` |

### How to read this notebook
Each section follows the same template:
1. **"You are here"** — the diagram with the current topic highlighted
2. **The question** — what problem this piece solves (1-2 sentences)
3. **Math + intuition** — formula with each symbol labeled, plus a plain-English analogy
4. **Visualization** — heatmaps / curves / matrix art on real tensors
5. **Code** — import from the `.py` files and run
6. **Shape trace** — a table showing how tensor shapes evolve

## Setup

Run this cell once. It adds `part_1/` to the path so we can `import` from its modules, and it sets up common imports.

In [ ]:
import sys, pathlib
# This notebook lives inside part_1/. Add its directory to sys.path so local imports work.
NB_DIR = pathlib.Path().resolve()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

print("Setup complete. Working directory:", NB_DIR)

---
## 1.1 Positional Encoding

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]

    pe["1.1 Positional Encoding<br>add position info"]
    ln1["1.5 LayerNorm 1"]
    mha["1.3/1.4 Multi-Head Attention<br>(Q, K, V + causal mask)"]
    add1["+ residual"]
    ln2["1.5 LayerNorm 2"]
    ffn["1.5 Feed-Forward<br>Linear -> GELU -> Linear"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> pe --> ln1 --> mha --> add1
    pe --> add1
    add1 --> ln2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style pe fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ln1 fill:#e8e8e8,stroke:#bbb,color:#777
    style mha fill:#e8e8e8,stroke:#bbb,color:#777
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style ln2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

Self-attention (we'll build it in section 1.2) has no notion of order. If you shuffle the input tokens, attention produces the *same set of outputs* in shuffled order. So the model literally cannot tell "the cat sat on the mat" from "mat the on sat cat the" without extra help.

**Positional encoding** is that extra help: a vector added to each token's embedding that encodes its position `t = 0, 1, 2, ...`. Two flavors:

| | Learned | Sinusoidal (original paper) |
|---|---|---|
| What | An `nn.Embedding(max_len, d_model)` table — each position gets a trainable vector | Fixed sine/cosine waves at different frequencies |
| Pro | Simple, often works better empirically | No parameters, extrapolates to longer sequences |
| Con | Can't extrapolate past `max_len` seen in training | Slightly less expressive |

### The sinusoidal formula

For position `pos` and dimension `i`:

$$
\mathrm{PE}(pos, 2i)   = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$
$$
\mathrm{PE}(pos, 2i+1) = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

**Why this weird formula?**
- Even dims use sine, odd dims use cosine — alternating
- The denominator `10000^(2i/d_model)` makes each dim oscillate at a different wavelength: low-index dims wiggle fast (sensitive to small position differences), high-index dims wiggle slowly (sensitive to large differences)
- Result: the dot product between `PE(pos)` and `PE(pos+k)` depends only on `k`, not on `pos` — so the model can learn to reason about *relative* positions

### Visualization 1: the sinusoidal PE as a heatmap

Each row = one position. Each column = one embedding dimension. The classic striped pattern emerges: low dims (left) change fast between rows, high dims (right) change slowly.

In [ ]:
from pos_encoding import SinusoidalPositionalEncoding

max_len, d_model = 64, 64
spe = SinusoidalPositionalEncoding(max_len=max_len, d_model=d_model)
pe_matrix = spe.pe.numpy()  # (64, 64)

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(pe_matrix, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel("embedding dimension (i)")
ax.set_ylabel("position (pos)")
ax.set_title("Sinusoidal PE: each row = one position, each column = one dim")
fig.colorbar(im, ax=ax, label="value")
plt.tight_layout()
plt.show()

### Visualization 2: a few dimensions as waves

Plotting dimensions 0, 4, 16, 32 shows how the frequency decreases as dimension index grows. Dim 0 is a fast wave, dim 32 barely changes across 64 positions.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for d in [0, 4, 16, 32]:
    ax.plot(pe_matrix[:, d], label=f"dim {d}")
ax.set_xlabel("position")
ax.set_ylabel("PE value")
ax.set_title("Sinusoidal PE — different dims oscillate at different frequencies")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Code — adding PE to a fake batch of embeddings

`pos_encoding.py` wraps the formula in an `nn.Module`. Its `forward(x)` adds the positional matrix to the input embeddings and returns the sum.

In [ ]:
# Pretend we already have token embeddings (B=2, T=10, d_model=64)
x = torch.randn(2, 10, 64)
x_with_pos = spe(x)
print("input shape: ", x.shape)
print("output shape:", x_with_pos.shape)
print("first 5 values of token 0 position 0 before PE:", x[0, 0, :5].numpy())
print("first 5 values of token 0 position 0 after  PE:", x_with_pos[0, 0, :5].numpy())

### Shape trace

| Stage | Shape | Meaning |
|-------|-------|---------|
| Input `x` | `(B, T, d_model)` | token embeddings |
| PE matrix | `(T, d_model)` | looked up from a `(max_len, d_model)` table |
| Output | `(B, T, d_model)` | `x + pe` — broadcast over batch |

**Key takeaway:** PE is added, not concatenated. The model learns to use part of the embedding capacity for position.

---
## 1.2 Self-attention from first principles (NumPy)

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]

    pe["1.1 Positional Encoding<br>add position info"]
    ln1["1.5 LayerNorm 1"]
    mha["1.3/1.4 Multi-Head Attention<br>(Q, K, V + causal mask)"]
    add1["+ residual"]
    ln2["1.5 LayerNorm 2"]
    ffn["1.5 Feed-Forward<br>Linear -> GELU -> Linear"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> pe --> ln1 --> mha --> add1
    pe --> add1
    add1 --> ln2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style pe fill:#e8e8e8,stroke:#bbb,color:#777
    style ln1 fill:#e8e8e8,stroke:#bbb,color:#777
    style mha fill:#fdebd0,stroke:#b35900,stroke-width:3px,color:#000
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style ln2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

Each token needs to gather information from other tokens. Which tokens? How much from each? **Attention** answers this by comparing every pair of tokens through *queries* and *keys*, then mixing their *values* weighted by similarity.

### The database analogy

Imagine a query-able database:

| Concept | Database | Attention |
|---|---|---|
| **Query** (Q) | "what am I looking for?" | token's question — a vector |
| **Key** (K) | the record's label/index | token's advertised content — a vector |
| **Value** (V) | what you retrieve if the key matches | what the token actually shares — a vector |

In classical databases, you check `query == key` (exact match). In attention, we check `similarity(query, key)` (soft match via dot product), then take a weighted average of values.

### The formula

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
$$

Step by step:
1. **`Q, K, V = X W_q, X W_k, X W_v`** — project input into three learned spaces
2. **`S = Q K^\top`** — scores: how much each query matches each key (dot product = similarity)
3. **`S = S / sqrt(d_k)`** — scale down so softmax doesn't saturate (values don't blow up as `d_k` grows)
4. **`S[i, j] = -infinity for j > i`** — causal mask: token `i` can't look at future tokens `j > i`
5. **`W = softmax(S)`** — normalize each row to a probability distribution over past tokens
6. **`Out = W V`** — weighted sum of values

### Tiny concrete example (T=3, d_model=4, d_k=2)

We'll use `attn_numpy_demo.py`'s hand-picked numbers so the matrix shapes are walkable on paper.

In [ ]:
# Toy input: 1 batch, 3 tokens, 4-dim embeddings
X = np.array([[[0.1, 0.2, 0.3, 0.4],
               [0.5, 0.4, 0.3, 0.2],
               [0.0, 0.1, 0.0, 0.1]]], dtype=np.float32)

# Fixed Q, K, V projection matrices (would be learned in a real model)
Wq = np.array([[ 0.2, -0.1], [ 0.0,  0.1], [ 0.1,  0.2], [-0.1,  0.0]], dtype=np.float32)
Wk = np.array([[ 0.1,  0.1], [ 0.0, -0.1], [ 0.2,  0.0], [ 0.0,  0.2]], dtype=np.float32)
Wv = np.array([[ 0.1,  0.0], [-0.1,  0.1], [ 0.2, -0.1], [ 0.0,  0.2]], dtype=np.float32)

Q = X @ Wq  # (1, 3, 2)
K = X @ Wk
V = X @ Wv
print("Q (3 tokens x 2 dims):\n", Q[0])
print("\nK:\n", K[0])
print("\nV:\n", V[0])

### Visualization: the full attention pipeline as four matrices

Left to right: **Q, K, V** as heatmaps, then the **scores `Q @ Kᵀ`**, then the **weights after softmax + causal mask**, then the **output `W @ V`**. Each row of the weights matrix sums to 1 (it's a probability distribution over past tokens).

In [ ]:
scale = 1.0 / np.sqrt(Q.shape[-1])
scores = (Q @ K.transpose(0, 2, 1)) * scale  # (1, 3, 3)

# Causal mask: upper triangle set to -inf
mask = np.triu(np.ones((3, 3), dtype=bool), k=1)
scores_masked = np.where(mask, -1e9, scores[0])

# Softmax row-wise
weights = np.exp(scores_masked - scores_masked.max(axis=-1, keepdims=True))
weights = weights / weights.sum(axis=-1, keepdims=True)
out = weights @ V[0]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, mat, title in zip(
    axes,
    [scores[0], scores_masked, weights, out],
    ["scores = QK^T / sqrt(d_k)", "after causal mask (-inf above diag)", "weights = softmax(scores)", "output = weights @ V"],
):
    im = ax.imshow(mat, aspect='auto', cmap='RdBu_r')
    ax.set_title(title, fontsize=10)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if val < -1e6:
                text = "-inf"
            else:
                text = f"{val:.2f}"
            ax.text(j, i, text, ha='center', va='center', fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

### Reading the weights matrix

Each row of `weights` is "how much does token `i` attend to each earlier token `j`?"
- **Row 0** (first token): `[1, 0, 0]` — can only look at itself
- **Row 1** (second token): `[w1, w2, 0]` — mixes tokens 0 and 1
- **Row 2** (third token): `[w1, w2, w3]` — mixes all three

This triangular structure is the signature of **causal** attention. The model can't peek at the future — essential for next-token prediction.

### Shape trace

| Stage | Shape | Formula |
|-------|-------|---------|
| Input `X` | `(B, T, d_model)` | tokens after embedding + PE |
| `Q, K, V` | each `(B, T, d_k)` | `X @ W_q`, etc. |
| `Q @ K^T` | `(B, T, T)` | similarity between all token pairs |
| `softmax` result | `(B, T, T)` | each row sums to 1 |
| Output | `(B, T, d_k)` | `weights @ V` |

---
## 1.3 Single Head in PyTorch

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]

    pe["1.1 Positional Encoding<br>add position info"]
    ln1["1.5 LayerNorm 1"]
    mha["1.3/1.4 Multi-Head Attention<br>(Q, K, V + causal mask)"]
    add1["+ residual"]
    ln2["1.5 LayerNorm 2"]
    ffn["1.5 Feed-Forward<br>Linear -> GELU -> Linear"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> pe --> ln1 --> mha --> add1
    pe --> add1
    add1 --> ln2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style pe fill:#e8e8e8,stroke:#bbb,color:#777
    style ln1 fill:#e8e8e8,stroke:#bbb,color:#777
    style mha fill:#fdebd0,stroke:#b35900,stroke-width:3px,color:#000
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style ln2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

### From NumPy to `nn.Module`

Same math, now wrapped in a PyTorch module so gradients flow automatically and we can train the `W_q`, `W_k`, `W_v` matrices. Here's the structure of `single_head.py`:

```python
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        self.q = nn.Linear(d_model, d_k, bias=False)  # W_q
        self.k = nn.Linear(d_model, d_k, bias=False)  # W_k
        self.v = nn.Linear(d_model, d_k, bias=False)  # W_v

    def forward(self, x):
        q = self.q(x)          # (B, T, d_k)
        k = self.k(x)
        v = self.v(x)
        scale = 1.0 / math.sqrt(q.size(-1))
        attn = q @ k.transpose(-2, -1) * scale     # (B, T, T)
        attn = attn.masked_fill(causal_mask, -inf) # hide future
        w = F.softmax(attn, dim=-1)
        out = w @ v                                 # (B, T, d_k)
        return out, w
```

Three differences from the NumPy version:
1. **`nn.Linear(..., bias=False)`** — creates learnable weights (not hand-picked)
2. **`masked_fill`** — elegant way to apply the causal mask in PyTorch
3. **Returns the weights too** — so we can visualize what the head is doing

### Run it and check the shapes

In [ ]:
from single_head import SingleHeadSelfAttention

torch.manual_seed(0)
head = SingleHeadSelfAttention(d_model=32, d_k=16, trace_shapes=True)
print("parameters:", sum(p.numel() for p in head.parameters()), "(3 weight matrices of 32*16)")

x = torch.randn(2, 8, 32)  # (B=2, T=8, d_model=32)
out, w = head(x)
print("output:", out.shape)
print("weights:", w.shape)

### Visualization: the causal mask

The causal mask has `True` (blocked) above the diagonal and `False` (allowed) on/below. `masked_fill` replaces blocked positions with `-inf`, which become 0 after softmax. Here's the `(8, 8)` mask for our `T=8` example.

In [ ]:
from attn_mask import causal_mask

m = causal_mask(8).squeeze().numpy()  # (8, 8) bool
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(m.astype(int), cmap='Reds', aspect='equal')
ax.set_xlabel("key position (what I look at)")
ax.set_ylabel("query position (who's looking)")
ax.set_title("Causal mask — True (red) = blocked, False (white) = allowed")
for i in range(8):
    for j in range(8):
        ax.text(j, i, "X" if m[i, j] else "•", ha='center', va='center',
                color='white' if m[i, j] else 'black')
plt.tight_layout()
plt.show()

### Shape trace

| Stage | Shape | Notes |
|-------|-------|-------|
| Input `x` | `(B, T, d_model)` | usually `d_model = 512` or `768` |
| `q, k, v` | `(B, T, d_k)` | `d_k` is usually `d_model / n_head` |
| `attn = q @ k.T * scale` | `(B, T, T)` | one score per query-key pair |
| `w = softmax(attn)` | `(B, T, T)` | row-stochastic |
| `out = w @ v` | `(B, T, d_k)` | each token's new representation |

---
## 1.4 Multi-Head Attention

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]

    pe["1.1 Positional Encoding<br>add position info"]
    ln1["1.5 LayerNorm 1"]
    mha["1.3/1.4 Multi-Head Attention<br>(Q, K, V + causal mask)"]
    add1["+ residual"]
    ln2["1.5 LayerNorm 2"]
    ffn["1.5 Feed-Forward<br>Linear -> GELU -> Linear"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> pe --> ln1 --> mha --> add1
    pe --> add1
    add1 --> ln2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style pe fill:#e8e8e8,stroke:#bbb,color:#777
    style ln1 fill:#e8e8e8,stroke:#bbb,color:#777
    style mha fill:#fdebd0,stroke:#b35900,stroke-width:3px,color:#000
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style ln2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

A single attention head can only learn *one* pattern of "who should attend to whom." But language has many patterns happening simultaneously:
- Head A might learn "attend to the previous word" (local)
- Head B might learn "attend to the subject of the sentence" (syntactic)
- Head C might learn "attend to punctuation" (structural)

**Multi-head attention** runs `n_head` independent attention heads in parallel, then concatenates their outputs. Each head works in a smaller subspace (`d_head = d_model / n_head`) so total compute is about the same as one big head — but the model can learn diverse attention patterns.

### The split / concat diagram

```
Input x : (B, T, d_model=64)
    |
    v
Wqkv : Linear(64, 3*64) -> (B, T, 192)
    |
    v
view + split into 3 chunks of 64 -> q, k, v each (B, T, 64)
    |
    v
reshape (B, T, n_head=4, d_head=16) + transpose -> each (B, 4, T, 16)
    |
    v
attention within each head independently -> (B, 4, T, 16)
    |
    v
transpose + reshape back to (B, T, 64)   <- concatenate heads
    |
    v
output projection Linear(64, 64) -> (B, T, 64)
```

**Why the combined `W_qkv` matrix?** It's just an efficiency trick — one matmul of size `(d_model, 3*d_model)` instead of three matmuls of size `(d_model, d_model)`. Mathematically identical, faster on GPU.

### Run it and trace the shapes

In [ ]:
from multi_head import MultiHeadSelfAttention

torch.manual_seed(0)
mha = MultiHeadSelfAttention(d_model=64, n_head=4, trace_shapes=True)
x = torch.randn(1, 10, 64)
out, w = mha(x)
print("\nfinal out:", out.shape)
print("final weights:", w.shape, "  # (B, n_head, T, T)")

### Visualization: per-head attention heatmaps

Each head learns a different pattern. Below we run MHA on a random input and plot the `(T, T)` attention matrix for each of the 4 heads. Even on random weights, the heads already produce different patterns — after training on text, these patterns become meaningful.

In [ ]:
# The weights tensor is (B, n_head, T, T). Plot one per head.
w_np = w.detach().numpy()  # (1, 4, 10, 10)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for h, ax in enumerate(axes):
    im = ax.imshow(w_np[0, h], aspect='auto', cmap='Blues', vmin=0, vmax=w_np[0, h].max())
    ax.set_title(f"Head {h}")
    ax.set_xlabel("key pos")
    if h == 0:
        ax.set_ylabel("query pos")
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.suptitle("Attention weights per head (causal, random init)", y=1.02)
plt.tight_layout()
plt.show()

### Visualization: splitting heads visually

Here's what reshape + transpose actually does to the `q` tensor. Imagine `d_model = 8` split into `n_head = 4` heads of `d_head = 2`. Each color block = one head's slice.

In [ ]:
d_model, n_head = 8, 4
d_head = d_model // n_head

# Simulate: one row of q for a token, before and after the head split
q_row = np.arange(d_model)  # [0, 1, 2, 3, 4, 5, 6, 7]
q_split = q_row.reshape(n_head, d_head)  # shape (4, 2)

fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
axes[0].imshow(q_row.reshape(1, -1), aspect='auto', cmap='tab20')
axes[0].set_title("Before split: 1 big vector of 8 dims")
axes[0].set_xticks(range(d_model)); axes[0].set_yticks([])
for i in range(d_model):
    axes[0].text(i, 0, str(i), ha='center', va='center', color='white', fontweight='bold')

axes[1].imshow(q_split, aspect='auto', cmap='tab20')
axes[1].set_title("After split: 4 heads of 2 dims each")
axes[1].set_xticks(range(d_head)); axes[1].set_yticks(range(n_head))
axes[1].set_xlabel("d_head"); axes[1].set_ylabel("head")
for i in range(n_head):
    for j in range(d_head):
        axes[1].text(j, i, str(q_split[i, j]), ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

### Shape trace (B=1, T=10, d_model=64, n_head=4, d_head=16)

| Stage | Shape | Operation |
|-------|-------|-----------|
| Input `x` | `(1, 10, 64)` | — |
| After `W_qkv` | `(1, 10, 192)` | one Linear: `d_model -> 3*d_model` |
| After `view(B, T, 3, H, d_h)` | `(1, 10, 3, 4, 16)` | split last dim into `3 x n_head x d_head` |
| After `unbind(dim=2)` | each `(1, 10, 4, 16)` | split into q, k, v |
| After `transpose(1, 2)` | each `(1, 4, 10, 16)` | move head dim forward |
| `q @ k.T * scale` | `(1, 4, 10, 10)` | one `(T, T)` matrix **per head** |
| `softmax(...)` | `(1, 4, 10, 10)` | row-stochastic per head |
| `w @ v` | `(1, 4, 10, 16)` | weighted values per head |
| `transpose + view` | `(1, 10, 64)` | concatenate heads back together |
| Output proj `W_o` | `(1, 10, 64)` | learn how to combine head outputs |

**Key insight:** all heads run in parallel on the same input — they're not sequential, they're concatenated.

---
## 1.5a Feed-Forward Network (FFN)

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]

    pe["1.1 Positional Encoding<br>add position info"]
    ln1["1.5 LayerNorm 1"]
    mha["1.3/1.4 Multi-Head Attention<br>(Q, K, V + causal mask)"]
    add1["+ residual"]
    ln2["1.5 LayerNorm 2"]
    ffn["1.5 Feed-Forward<br>Linear -> GELU -> Linear"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> pe --> ln1 --> mha --> add1
    pe --> add1
    add1 --> ln2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style pe fill:#e8e8e8,stroke:#bbb,color:#777
    style ln1 fill:#e8e8e8,stroke:#bbb,color:#777
    style mha fill:#e8e8e8,stroke:#bbb,color:#777
    style add1 fill:#e8e8e8,stroke:#bbb,color:#777
    style ln2 fill:#e8e8e8,stroke:#bbb,color:#777
    style ffn fill:#fff3c4,stroke:#b38600,stroke-width:3px,color:#000
    style add2 fill:#e8e8e8,stroke:#bbb,color:#777
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

### The question

After MHA, each token has *gathered* information from the other tokens. Now each token needs to *process* that information. That's what the FFN does — it applies the same small neural network to each token position **independently** (no cross-token mixing).

Think of it as: "attention = communication, FFN = computation."

### The structure

```
Input (B, T, d_model)
   |
   v
Linear(d_model, 4*d_model)   <- expand to 4x wider
   |
   v
GELU                         <- non-linearity
   |
   v
Linear(4*d_model, d_model)   <- project back down
   |
   v
Dropout
   |
   v
Output (B, T, d_model)
```

**Why expand 4x?** The expansion gives more capacity to learn non-linear transformations. It's the largest source of parameters in a Transformer — typically ~2/3 of all parameters live in FFN layers.

**Why GELU, not ReLU?** GELU is smooth — its derivative is continuous everywhere, which helps gradient flow. At large positive x it behaves like ReLU (passes through), at large negative x it's zero, but around zero it's smoothly curved instead of a sharp kink.

### Visualization: GELU vs ReLU

In [ ]:
x = torch.linspace(-4, 4, 200)
relu = F.relu(x)
gelu = F.gelu(x)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x.numpy(), relu.numpy(), label='ReLU', linewidth=2)
ax.plot(x.numpy(), gelu.numpy(), label='GELU', linewidth=2)
ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
ax.legend()
ax.set_xlabel('x'); ax.set_ylabel('activation(x)')
ax.set_title('GELU is smooth, ReLU has a sharp kink at zero')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Run it

In [ ]:
from ffn import FeedForward

ffn = FeedForward(d_model=64, mult=4)
print("parameters:", sum(p.numel() for p in ffn.parameters()))
print(f"  breakdown: 64*256 (expand) + 256 (bias) + 256*64 (project) + 64 (bias) = {64*256 + 256 + 256*64 + 64}")

x = torch.randn(2, 10, 64)
y = ffn(x)
print("\ninput: ", x.shape)
print("output:", y.shape, "  # same as input — FFN preserves shape")

### Shape trace

| Stage | Shape | Operation |
|-------|-------|-----------|
| Input `x` | `(B, T, d_model)` | token representations after MHA |
| After first `Linear` | `(B, T, 4*d_model)` | expand |
| After `GELU` | `(B, T, 4*d_model)` | non-linearity (same shape) |
| After second `Linear` | `(B, T, d_model)` | project back |

**The FFN is applied to each token independently** — no communication between positions. That's attention's job.

---
## 1.5b LayerNorm + Residual Connections

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]

    pe["1.1 Positional Encoding<br>add position info"]
    ln1["1.5 LayerNorm 1"]
    mha["1.3/1.4 Multi-Head Attention<br>(Q, K, V + causal mask)"]
    add1["+ residual"]
    ln2["1.5 LayerNorm 2"]
    ffn["1.5 Feed-Forward<br>Linear -> GELU -> Linear"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> pe --> ln1 --> mha --> add1
    pe --> add1
    add1 --> ln2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8e8e8,stroke:#bbb,color:#777
    style pe fill:#e8e8e8,stroke:#bbb,color:#777
    style ln1 fill:#e8daef,stroke:#6c3483,stroke-width:3px,color:#000
    style mha fill:#e8e8e8,stroke:#bbb,color:#777
    style add1 fill:#fdd,stroke:#c33,stroke-width:3px,color:#000
    style ln2 fill:#e8daef,stroke:#6c3483,stroke-width:3px,color:#000
    style ffn fill:#e8e8e8,stroke:#bbb,color:#777
    style add2 fill:#fdd,stroke:#c33,stroke-width:3px,color:#000
    style output fill:#e8e8e8,stroke:#bbb,color:#777
```

These are the glue that makes deep Transformers trainable. Every sublayer (MHA, FFN) is wrapped like this:

```python
x = x + sublayer(LayerNorm(x))
```

Two things happening:
1. **LayerNorm** — normalize the activations so they don't drift
2. **Residual connection** (the `x +`) — provide a "highway" for gradients

### LayerNorm: normalize per token, across features

For each token (one row of the `(B, T, d_model)` tensor), compute:

$$
\text{LN}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta
$$

where `μ, σ²` are the mean and variance **across the `d_model` features of that token** (not across batch, unlike BatchNorm). `γ, β` are learnable scale + shift.

**Why LayerNorm and not BatchNorm?**
- LayerNorm doesn't depend on batch statistics — works the same at train and test time
- Works for variable-length sequences (batch sizes of 1 are fine)
- Plays nicely with attention, which expects each token to be a self-contained unit

### Residual connection: the gradient highway

`x = x + sublayer(...)` means:
- The **forward** pass has two paths: through the sublayer, and straight through the `+`
- The **backward** pass also has two paths — gradient can skip the sublayer entirely via the `+`
- Even if the sublayer "kills" the gradient (common in deep nets), the skip path keeps it flowing

Without residuals, training a 12-layer (let alone 96-layer) Transformer is nearly impossible.

### Pre-Norm vs Post-Norm

The original Transformer paper (2017) used **Post-Norm**: `x = LN(x + sublayer(x))`.
Modern GPTs use **Pre-Norm**: `x = x + sublayer(LN(x))` — more stable to train, that's what we use here.

Notice in the diagram: LN comes *before* MHA and FFN, not after.

### Visualization: LayerNorm in action on a tiny tensor

Pick 4 tokens, each with 8 features. Before LN: arbitrary values with some mean and spread. After LN: each row has mean≈0 and std≈1.

In [ ]:
torch.manual_seed(0)
x = torch.randn(4, 8) * 2 + 3  # (4 tokens, 8 features)
ln = nn.LayerNorm(8, elementwise_affine=False)  # no gamma/beta for visualization clarity
y = ln(x)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, mat, title in zip(axes, [x.numpy(), y.numpy()], ["Before LayerNorm", "After LayerNorm"]):
    im = ax.imshow(mat, aspect='auto', cmap='RdBu_r', vmin=-3, vmax=3)
    ax.set_title(title + f"\nrow means: {mat.mean(1).round(2)}\nrow stds:  {mat.std(1).round(2)}",
                 fontsize=9, loc='left')
    ax.set_xlabel("feature dim"); ax.set_ylabel("token")
    for i in range(4):
        for j in range(8):
            ax.text(j, i, f"{mat[i,j]:.1f}", ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

### Visualization: why residual connections are a "highway"

Without residuals, the gradient flowing from output to input must survive every sublayer. With residuals, it has a direct path.

```
Without residual:  gradient must pass through each sublayer untouched
    loss <-- sublayer_N <-- ... <-- sublayer_1 <-- input

With residual:    gradient has a skip path around each sublayer
    loss <-- (+ skip) <-- ... <-- (+ skip) <-- input
```

Even if each sublayer multiplies gradients by `0.9`, after 24 layers the signal is `0.9^24 ≈ 0.08` — vanishing. With residual skips, the unchanged path is always available.

---
## 1.6 The Full Transformer Block

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]

    pe["1.1 Positional Encoding<br>add position info"]
    ln1["1.5 LayerNorm 1"]
    mha["1.3/1.4 Multi-Head Attention<br>(Q, K, V + causal mask)"]
    add1["+ residual"]
    ln2["1.5 LayerNorm 2"]
    ffn["1.5 Feed-Forward<br>Linear -> GELU -> Linear"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> pe --> ln1 --> mha --> add1
    pe --> add1
    add1 --> ln2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8f4fd,stroke:#2874a6,stroke-width:3px,color:#000
    style pe fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ln1 fill:#e8daef,stroke:#6c3483,stroke-width:3px,color:#000
    style mha fill:#fdebd0,stroke:#b35900,stroke-width:3px,color:#000
    style add1 fill:#fdd,stroke:#c33,stroke-width:3px,color:#000
    style ln2 fill:#e8daef,stroke:#6c3483,stroke-width:3px,color:#000
    style ffn fill:#fff3c4,stroke:#b38600,stroke-width:3px,color:#000
    style add2 fill:#fdd,stroke:#c33,stroke-width:3px,color:#000
    style output fill:#e8f4fd,stroke:#2874a6,stroke-width:3px,color:#000
```

### Assembly

From `block.py`:

```python
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_head, dropout=0.0):
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_head, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = FeedForward(d_model, mult=4, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))[0]   # communicate
        x = x + self.ffn (self.ln2(x))      # compute
        return x
```

Two lines for the whole block, after all that work! The pattern is `x = x + sublayer(LN(x))` applied twice — once for attention, once for FFN.

### Full forward pass with shape trace

In [ ]:
from block import TransformerBlock

torch.manual_seed(0)
block = TransformerBlock(d_model=64, n_head=4)
print(f"Total parameters: {sum(p.numel() for p in block.parameters()):,}")
print()

# Count parameters per submodule
for name, mod in block.named_children():
    n_params = sum(p.numel() for p in mod.parameters())
    print(f"  {name:10s} : {n_params:>7,} params")

In [ ]:
# Trace one forward pass with shape prints at every intermediate step
x = torch.randn(2, 10, 64)  # (B=2, T=10, d_model=64)
print("input x:            ", x.shape)

h = block.ln1(x)
print("after ln1:          ", h.shape)

attn_out, attn_w = block.attn(h)
print("after attention:    ", attn_out.shape, "  (attention weights:", attn_w.shape, ")")

x1 = x + attn_out
print("after residual 1:   ", x1.shape)

h2 = block.ln2(x1)
print("after ln2:          ", h2.shape)

ff_out = block.ffn(h2)
print("after FFN:          ", ff_out.shape)

x2 = x1 + ff_out
print("final output:       ", x2.shape)

### Stacking blocks into a model

A full GPT model is: token embedding + position encoding + **N Transformer blocks stacked** + final LayerNorm + linear head to vocab size. That's Part 2. But you've now built the fundamental unit that gets repeated.

```
Tokens -> Embed -> + PE -> [Block] x N -> LN -> Linear -> logits -> softmax -> next-token probs
```

---
## Recap

```mermaid
graph TD
    input["Input tokens<br>(B, T, d_model)"]

    pe["1.1 Positional Encoding<br>add position info"]
    ln1["1.5 LayerNorm 1"]
    mha["1.3/1.4 Multi-Head Attention<br>(Q, K, V + causal mask)"]
    add1["+ residual"]
    ln2["1.5 LayerNorm 2"]
    ffn["1.5 Feed-Forward<br>Linear -> GELU -> Linear"]
    add2["+ residual"]
    output["Block output<br>(B, T, d_model)"]

    input --> pe --> ln1 --> mha --> add1
    pe --> add1
    add1 --> ln2 --> ffn --> add2
    add1 --> add2
    add2 --> output

    style input fill:#e8f4fd,stroke:#2874a6,stroke-width:3px,color:#000
    style pe fill:#d5f5e3,stroke:#28a745,stroke-width:3px,color:#000
    style ln1 fill:#e8daef,stroke:#6c3483,stroke-width:3px,color:#000
    style mha fill:#fdebd0,stroke:#b35900,stroke-width:3px,color:#000
    style add1 fill:#fdd,stroke:#c33,stroke-width:3px,color:#000
    style ln2 fill:#e8daef,stroke:#6c3483,stroke-width:3px,color:#000
    style ffn fill:#fff3c4,stroke:#b38600,stroke-width:3px,color:#000
    style add2 fill:#fdd,stroke:#c33,stroke-width:3px,color:#000
    style output fill:#e8f4fd,stroke:#2874a6,stroke-width:3px,color:#000
```

You've built every piece of the Transformer block:

| Section | Piece | Purpose |
|---------|-------|---------|
| **1.1** | Positional encoding | tell the model where each token is |
| **1.2** | Self-attention math | content-based information gathering |
| **1.3** | Single head in PyTorch | same math, now trainable |
| **1.4** | Multi-head attention | multiple attention patterns in parallel |
| **1.5a** | Feed-forward | per-token computation |
| **1.5b** | LayerNorm + residuals | make deep networks trainable |
| **1.6** | Full block | assemble into one unit |

### What's next — Part 2

The block you just built is the core repeated unit. To turn it into a working language model you need:
- **Token embedding** — map vocabulary IDs to dense vectors
- **Stack of blocks** — typically 6 / 12 / 24 / 96 layers
- **LM head** — project the final `d_model` back to vocab size
- **Loss function** — cross-entropy on next-token prediction
- **Training loop** — data batching, optimizer, eval
- **Sampling** — temperature, top-k, top-p

All of that is Part 2. But the hardest conceptual lifting is done.